## 🔺️ Shape Image Classification

Given *images of shapes*, let's try to predict which **shape** is present in a given image.

We will use a TensorFlow/Keras convolutional neural network to make our predictions.

Data source: https://www.kaggle.com/datasets/cactus3/basicshapes

### Importing Libraries

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import os.path

from sklearn.model_selection import train_test_split

import tensorflow as tf

I0000 00:00:1793803854.998121   15309 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793803855.441746   15309 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793803858.249060   15309 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
image_dir = Path("archive/shapes")

### Create a File DataFrame

In [11]:
filepaths = list(image_dir.glob(r'**/*.png'))
filepaths

[PosixPath('archive/shapes/triangles/drawing(29).png'),
 PosixPath('archive/shapes/triangles/drawing(80).png'),
 PosixPath('archive/shapes/triangles/drawing(50).png'),
 PosixPath('archive/shapes/triangles/drawing(14).png'),
 PosixPath('archive/shapes/triangles/drawing(20).png'),
 PosixPath('archive/shapes/triangles/drawing(7).png'),
 PosixPath('archive/shapes/triangles/drawing(77).png'),
 PosixPath('archive/shapes/triangles/drawing(56).png'),
 PosixPath('archive/shapes/triangles/drawing(38).png'),
 PosixPath('archive/shapes/triangles/drawing(35).png'),
 PosixPath('archive/shapes/triangles/drawing(60).png'),
 PosixPath('archive/shapes/triangles/drawing(34).png'),
 PosixPath('archive/shapes/triangles/drawing(81).png'),
 PosixPath('archive/shapes/triangles/drawing(93).png'),
 PosixPath('archive/shapes/triangles/drawing(79).png'),
 PosixPath('archive/shapes/triangles/drawing(36).png'),
 PosixPath('archive/shapes/triangles/drawing(73).png'),
 PosixPath('archive/shapes/triangles/drawing(23).

In [12]:
labels = list(map(lambda x: os.path.split(os.path.split(x)[0])[1], filepaths))

labels = pd.Series(labels, name='Label')
filepaths = pd.Series(filepaths, name='Filepath').astype(str)

image_df = pd.concat([filepaths, labels], axis=1)

image_df

,Filepath,Label
0,archive/shapes/triangles/drawing(29).png,triangles
1,archive/shapes/triangles/drawing(80).png,triangles
2,archive/shapes/triangles/drawing(50).png,triangles
3,archive/shapes/triangles/drawing(14).png,triangles
4,archive/shapes/triangles/drawing(20).png,triangles
...,...,...
295,archive/shapes/squares/drawing(74).png,squares
296,archive/shapes/squares/drawing(28).png,squares
297,archive/shapes/squares/drawing(11).png,squares
298,archive/shapes/squares/drawing(90).png,squares


In [13]:
train_df, test_df = train_test_split(image_df, train_size=0.7, shuffle=True, random_state=1)

In [14]:
train_df

,Filepath,Label
253,archive/shapes/squares/drawing(84).png,squares
19,archive/shapes/triangles/drawing(57).png,triangles
14,archive/shapes/triangles/drawing(79).png,triangles
91,archive/shapes/triangles/drawing(61).png,triangles
296,archive/shapes/squares/drawing(28).png,squares
...,...,...
203,archive/shapes/squares/drawing(14).png,squares
255,archive/shapes/squares/drawing(65).png,squares
72,archive/shapes/triangles/drawing(85).png,triangles
235,archive/shapes/squares/drawing(72).png,squares


In [15]:
test_df

,Filepath,Label
189,archive/shapes/circles/drawing(96).png,circles
123,archive/shapes/circles/drawing(26).png,circles
185,archive/shapes/circles/drawing(10).png,circles
213,archive/shapes/squares/drawing(93).png,squares
106,archive/shapes/circles/drawing(77).png,circles
...,...,...
181,archive/shapes/circles/drawing(49).png,circles
290,archive/shapes/squares/drawing(52).png,squares
244,archive/shapes/squares/drawing(87).png,squares
197,archive/shapes/circles/drawing(11).png,circles


### Loading Image Data

In [16]:
train_generator = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale = 1. / 255,
    validation_split = 0.2
)

test_generator = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale = 1. / 255
)

In [17]:
train_images = train_generator.flow_from_dataframe(
    dataframe = train_df,
    x_col = 'Filepath',
    y_col = 'Label',
    target_size = (28, 28),
    color_mode = 'grayscale',
    class_mode = 'categorical',
    batch_size = 32,
    shuffle = True,
    seed = 42,
    subset = 'training'
)

val_images = train_generator.flow_from_dataframe(
    dataframe = train_df,
    x_col = 'Filepath',
    y_col = 'Label',
    target_size = (28, 28),
    color_mode = 'grayscale',
    class_mode = 'categorical',
    batch_size = 32,
    shuffle = True,
    seed = 42,
    subset = 'validation'
)

test_images = test_generator.flow_from_dataframe(
    dataframe = test_df,
    x_col = 'Filepath',
    y_col = 'Label',
    target_size = (28, 28),
    color_mode = 'grayscale',
    class_mode = 'categorical',
    batch_size = 32,
    shuffle = False
)

Found 168 validated image filenames belonging to 3 classes.
Found 42 validated image filenames belonging to 3 classes.
Found 90 validated image filenames belonging to 3 classes.


### Training

In [28]:
inputs = tf.keras.Input(shape=(28, 28, 1))
x = tf.keras.layers.Conv2D(filters = 6, kernel_size = (3, 3), activation='relu')(inputs)
x = tf.keras.layers.MaxPooling2D()(x)
x = tf.keras.layers.Conv2D(filters = 16, kernel_size = (3, 3), activation='relu')(x)
x = tf.keras.layers.MaxPooling2D()(x)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
outputs = tf.keras.layers.Dense(3, activation='softmax')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer = 'adam',
    loss = 'categorical_crossentropy',
    metrics = ['accuracy']
)

history = model.fit(
    train_images,
    validation_data = val_images,
    epochs = 100,
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath = './model.weights.h5',
            save_best_only = True,
            save_weights_only = True
        )
    ]
)

Epoch 1/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 81ms/step - accuracy: 0.3274 - loss: 1.1096 - val_accuracy: 0.4048 - val_loss: 1.0920
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.3274 - loss: 1.1009 - val_accuracy: 0.4048 - val_loss: 1.0924
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3988 - loss: 1.0984 - val_accuracy: 0.3333 - val_loss: 1.0963
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3393 - loss: 1.0982 - val_accuracy: 0.3333 - val_loss: 1.1016
Epoch 5/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3393 - loss: 1.1002 - val_accuracy: 0.3333 - val_loss: 1.1009
Epoch 6/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3393 - loss: 1.0973 - val_accuracy: 0.2619 - val_loss: 1.1005
Epoch 7/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3333 - loss: 1.0994 - val_accuracy: 0.2619 - val_loss: 1.1019
Epoch 8/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.3333 - loss: 1.0991 - val_accuracy: 0.2619 - val_loss:

In [30]:
model.load_weights('./model.weights.h5')

### Results

In [31]:
results = model.evaluate(test_images, verbose=0)

print("    Loss: {:.4f}".format(results[0]))
print("Accuracy: {:.2f}%".format(results[1]*100))

    Loss: 0.6104
Accuracy: 62.22%
